# External validation of the frozen SEER model on AACR Project GENIE BPC CRC v2.0-public
**Project:** Explainable AI-Driven Survival Prediction in Colorectal Cancer (M.Tech, MANUU) — companion to `CRC_SEER_Survival_Prediction.ipynb`.

**Rules this notebook follows**
* The SEER model, its preprocessing, its calibrator and its threshold are **loaded, never refitted**. Nothing in this notebook trains or tunes on GENIE outcomes.
* GENIE files are merged only on verified keys (`record_id`, `ca_seq`), with merge cardinality checked.
* Coded GENIE values are **never guessed**: Step 3 prints every code, and Step 4 refuses to continue while any code is missing from the mapping table that you fill in from the official *BPC CRC v2.0 Analytic Data Guide*.
* **Privacy:** GENIE BPC data are under the AACR GENIE terms of access. Keep the CSVs on your own computer. Do not commit them to GitHub or upload them anywhere. This notebook saves only aggregated tables and figures.


## Step 0 — Configuration
`MODEL_TO_VALIDATE` selects which frozen SEER artifact is validated:
* `"core"` — the **transportable core model** (only predictors that exist in both SEER and GENIE; trained on SEER only in the SEER notebook, Step 17b). **Recommended primary analysis.**
* `"A"` — the diagnosis-time model (Set A). Four predictors are missing in GENIE, so it can only be run as a **stress test** (`ALLOW_STRESS_TEST = True`).
* `"B"` — the primary post-surgical model (Set B). Seven predictors are missing in GENIE: stress test only.


In [ ]:
import json, hashlib, warnings
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, joblib
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from sklearn.calibration import calibration_curve
from lifelines import KaplanMeierFitter
from lifelines.utils import concordance_index
warnings.filterwarnings("ignore")

SEED              = 42
GENIE_DIR         = Path("genie_bpc_crc_2.0")          # folder with the GENIE CSVs (NOT inside the git repo)
ART               = Path("artifacts")                  # frozen artifacts written by the SEER notebook (Step 17)
MODEL_TO_VALIDATE = "core"                             # "core" (recommended) | "A" | "B"
ALLOW_STRESS_TEST = False                              # must be True to run "A"/"B" with missing predictors
HORIZON           = 60                                 # months
N_BOOT            = 1000
CEA_WINDOW_DAYS   = (-90, 30)                          # CEA measurement window around diagnosis (pre-treatment), see Step 6
FIRST_PRIMARY_ONLY = True                             # mirror SEER: keep ca_seq 0 (only primary) or 1 (first of several) - verify ca_seq = NAACCR sequence number
HISTOLOGY_MISSING = "exclude"                         # "exclude" (primary analysis) | "modal" (sensitivity: assign the commonest SEER group)
OUTG = Path("outputs_genie"); (OUTG / "figures").mkdir(parents=True, exist_ok=True)
def savefig(name): plt.savefig(OUTG / "figures" / f"{name}.png", dpi=200, bbox_inches="tight")

## Step 1 — Load the frozen SEER model (no retraining)
Loads the model file, prints its SHA-256 hash (record it in the paper's supplement) and reads the metadata that defines the target, the threshold and the exact feature list. If the file is missing, stop: re-run Step 17 of the SEER notebook on your computer to export it.


In [ ]:
FILES = {"core": "crc_model_core_transportable.joblib", "A": "crc_model_setA_diagnosis.joblib", "B": "crc_model.joblib"}
meta = json.loads((ART / "model_meta.json").read_text())
model_path = ART / FILES[MODEL_TO_VALIDATE]
if not model_path.exists():
    raise FileNotFoundError(f"{model_path} not found - run the SEER notebook up to Step 17b first. Do NOT train a model here.")
model = joblib.load(model_path)
calibrator = joblib.load(ART / "crc_calibrator.joblib") if (MODEL_TO_VALIDATE == "B" and (ART / "crc_calibrator.joblib").exists()) else None
if MODEL_TO_VALIDATE == "core" and (ART / "crc_calibrator_core.joblib").exists():
    calibrator = joblib.load(ART / "crc_calibrator_core.joblib")
sha = hashlib.sha256(model_path.read_bytes()).hexdigest()

if MODEL_TO_VALIDATE == "B":      FEATS = meta["feature_cols"]
elif MODEL_TO_VALIDATE == "A":    FEATS = meta["setA_model"]["feature_cols"]
else:                             FEATS = meta["core_model"]["feature_cols"]
NUMERIC_SEER = meta["numeric"]; CATEG_SEER = meta["categorical"]
THRESHOLD = meta["threshold"]
print("Model file      :", model_path, "| sha256", sha[:16], "...")
print("Target          :", meta["target"])
print("Threshold       :", THRESHOLD, "applied to", meta["threshold_applies_to"])
print("Calibrator      :", "loaded" if calibrator is not None else "none")
print(f"Features ({len(FEATS)}):", FEATS)

## Step 2 — Inspect the GENIE files (schemas, missingness, identifiers)
Only the two main files are needed for the endpoint and most predictors. `tm_level_dataset.csv` supplies CEA and `cancer_panel_test_level_dataset.csv`
(the cancer-panel-test = sequencing report file) supplies the date of genomic sequencing, which matters for **delayed entry** (Step 5).
`cancer_level_dataset_non_index.csv` (other cancers of the same patients) is **not** used, so that no non-CRC cancer enters the cohort.


In [ ]:
REQUIRED = {"index": "cancer_level_dataset_index.csv", "patient": "patient_level_dataset.csv",
            "tm": "tm_level_dataset.csv", "cpt": "cancer_panel_test_level_dataset.csv"}
G = {}
for k, f in REQUIRED.items():
    p = GENIE_DIR / f
    if not p.exists(): raise FileNotFoundError(f"missing {p}")
    G[k] = pd.read_csv(p, low_memory=False)

def schema(df, name, keys):
    s = pd.DataFrame({"dtype": df.dtypes.astype(str), "n_missing": df.isna().sum(), "pct_missing": (100 * df.isna().mean()).round(1),
                      "n_unique": df.nunique()})
    print(f"\n=== {name}: {df.shape[0]:,} rows x {df.shape[1]} cols | key {keys}: {int(df.duplicated(keys).sum())} duplicate key rows "
          f"| exact duplicate rows: {int(df.duplicated().sum())} | patients: {df.record_id.nunique():,}")
    print("release_version:", df["release_version"].unique().tolist() if "release_version" in df else "n/a",
          "| cohort:", df["cohort"].unique().tolist() if "cohort" in df else "n/a", "| institutions:", df["institution"].value_counts().to_dict() if "institution" in df else "n/a")
    s.to_csv(OUTG / f"schema_{name}.csv")
    return s

sch = {"index": schema(G["index"], "index", ["record_id", "ca_seq"]),
       "patient": schema(G["patient"], "patient", ["record_id"]),
       "tm": schema(G["tm"], "tm", ["record_id", "tm_number"]),
       "cpt": schema(G["cpt"], "cpt", ["record_id", "ca_seq", "cpt_number"])}
display(sch["index"].loc[[c for c in ["age_dx", "ca_d_site", "naaccr_histology_cd", "naaccr_behavior_cd", "ca_grade", "best_ajcc_stage_cd", "stage_dx",
                                      "naaccr_path_t_cd", "naaccr_path_n_cd", "naaccr_path_m_cd", "naaccr_clin_t_cd", "naaccr_clin_n_cd", "naaccr_clin_m_cd",
                                      "naaccr_seer_sum_stage", "ca_crc_peri_inv", "ca_crc_td", "os_dx_status", "tt_os_dx_mos"] if c in sch["index"].index]])

## Step 3 — Merge relationships and the official meaning of every code
* `patient_level_dataset` must have one row per `record_id`; index cancers must be unique on (`record_id`, `ca_seq`). The merge is checked with `validate="m:1"`.
* The cell then prints every distinct value of each coded variable. **Fill the `CODEBOOK` in Step 4 from the official GENIE BPC CRC v2.0 Analytic Data Guide** (it is distributed with the data on Synapse). Do not fill it from memory or from this notebook's comments.


In [ ]:
idx, pat = G["index"].copy(), G["patient"].copy()
assert pat.record_id.is_unique, "patient_level_dataset has duplicate record_id"
assert not idx.duplicated(["record_id", "ca_seq"]).any(), "index file has duplicate (record_id, ca_seq)"
per_pt = idx.groupby("record_id").size()
print("Index CRC cancers per patient:", per_pt.value_counts().to_dict())
df = idx.merge(pat.drop(columns=[c for c in ["cohort", "institution", "release_version"] if c in pat]), on="record_id", how="left", validate="m:1", indicator=True)
print("Merge result:", df["_merge"].value_counts().to_dict()); assert (df["_merge"] == "both").all(); df = df.drop(columns="_merge")

CODED = ["os_dx_status", "stage_dx", "best_ajcc_stage_cd", "ca_grade", "naaccr_seer_sum_stage", "naaccr_path_t_cd", "naaccr_path_n_cd", "naaccr_path_m_cd",
         "naaccr_clin_t_cd", "naaccr_clin_n_cd", "naaccr_clin_m_cd", "ca_crc_peri_inv", "ca_crc_td", "ca_d_site", "naaccr_behavior_cd",
         "naaccr_sex_code", "naaccr_race_code_primary", "naaccr_ethnicity_code", "race_ethnicity", "crc_type", "ca_type"]
for c in CODED:
    if c in df: print(f"\n{c}:", df[c].value_counts(dropna=False).to_dict())
print("\nage_dx range:", df.age_dx.min(), "-", df.age_dx.max(), "| tt_os_dx_mos range:", df.tt_os_dx_mos.min(), "-", df.tt_os_dx_mos.max())

## Step 4 — Codebook (fill from the official data guide) and SEER-to-GENIE harmonisation
Each entry maps an **observed GENIE value** to the **SEER category the frozen model was trained on** (the SEER categories are read from `model_meta.json`).
`check_codebook()` stops the notebook if a GENIE value has no mapping or maps to a category the SEER model never saw.

The values below were taken from the **values actually present in the BPC CRC v2.0-public files** and checked against the
GENIE BPC CRC v2.0-public Analytic Data Guide and the NAACCR Data Dictionary (v26). Self-explanatory labels (e.g. "Male") are mapped directly.


In [ ]:
CODEBOOK = {
    # os_dx_status: 1 = Dead, 0 = Censored (GENIE BPC CRC v2.0-public Analytic Data Guide).
    "os_dx_status": {1: 1, 0: 0},
    "naaccr_sex_code": {"Male": "Male", "Female": "Female"},
    "ca_grade": {"I": "I", "II": "II", "III": "III", "IV": "IV"},           # blank (195) -> "Unknown", as SEER's unknown grade
    # NAACCR item #759 (SEER Summary Stage 2000, Data Dictionary v26): 1 localized, 2 regional direct extension, 3 regional nodes, 4 both, 5 regional NOS, 7 distant, 9 unstaged.
    # 5 -> "Unknown" because the SEER notebook mapped "Regional, NOS" to "Unknown" during training.
    "naaccr_seer_sum_stage": {1: "Localized", 2: "Regional (extension)", 3: "Regional (nodes)", 4: "Regional (both)", 5: "Unknown", 7: "Distant", 9: "Unknown"},
    "stage_dx": {"Stage I": "I", "Stage II": "II", "Stage III": "III", "Stage IV": "IV", "Stage I-III NOS": "Unknown"},
}
# Not in the core model, kept for stress tests. NOTE: ca_crc_peri_inv has NO explicit "absent" value in v2.0-public
# (present / not applicable / unknown / no histologic examination / blank), so "Absent" cannot be reconstructed.
PNI_MAP = {"Perineural invasion present": "Present"}
SITE_MAP = {"C18.0": "Cecum", "C18.1": "Appendix", "C18.2": "Ascending Colon", "C18.3": "Hepatic Flexure", "C18.4": "Transverse Colon",
            "C18.5": "Splenic Flexure", "C18.6": "Descending Colon", "C18.7": "Sigmoid Colon", "C18.8": "Large Intestine, NOS",
            "C18.9": "Large Intestine, NOS", "C19.9": "Rectosigmoid Junction", "C20.9": "Rectum"}   # ICD-O-3 topography -> SEER site recode

def check_codebook(frame):
    problems = []
    for col, mp in CODEBOOK.items():
        if col not in frame: problems.append(f"{col}: column not found"); continue
        unseen = sorted(set(frame[col].dropna().unique()) - set(mp), key=str)
        if unseen: problems.append(f"{col}: {len(unseen)} values without a mapping, e.g. {unseen[:6]}")
    return problems

problems = check_codebook(df)
print("\n".join(problems) if problems else "Codebook complete.")
if problems: raise SystemExit("Complete CODEBOOK from the official BPC CRC v2.0 data guide, then re-run this cell.")

## Step 5 — Eligible cohort and the five-year overall-survival endpoint
**Eligibility** (mirrors the SEER cohort): index CRC (`C18`–`C20`), invasive behaviour, age ≥ 20, known follow-up time. The attrition table is saved.

**Endpoint:** `time = tt_os_dx_mos` (months from diagnosis to death or last known alive) and `event = os_dx_status` mapped through the codebook.
* `y5 = 1` if time ≥ 60 months (alive at 5 years, whatever happened later)
* `y5 = 0` if event = death and time < 60
* `y5 = NaN` (**indeterminate**) if alive and time < 60 — **never** labelled a survivor; counted and reported.

**Why the binary endpoint alone is not enough in GENIE BPC.** Patients enter BPC only after tumour sequencing, so they had to survive from diagnosis
to sequencing (**left truncation / delayed entry**). Counting 5-year survival from diagnosis without accounting for this over-represents survivors.
The primary GENIE analysis is therefore **time-to-event with delayed entry at the first sequencing report** (Step 9); the binary metrics are reported
for the determinate patients and repeated in the subset sequenced within 6 months of diagnosis, where truncation is minimal.


In [ ]:
attr = [("Index CRC cancers (BPC CRC v2.0)", len(df))]
d = df.copy()
d["site3"] = d["ca_d_site"].astype(str).str[:3]
d = d[d.site3.isin(["C18", "C19", "C20"])];                                   attr.append(("ICD-O-3 site C18-C20", len(d)))
beh = pd.to_numeric(d["naaccr_behavior_cd"], errors="coerce")
d = d[beh.eq(3) | beh.isna()];                                                attr.append(("Invasive (ICD-O-3 behaviour 3; blank = curated diagnosis per BPC data guide, kept; no Stage 0 in stage_dx)", len(d)))
if FIRST_PRIMARY_ONLY:
    d = d[d.ca_seq.isin([0, 1])];                                             attr.append(("First or only primary (ca_seq 0/1), as in SEER", len(d)))
d = d.sort_values(["record_id", "ca_seq"]).drop_duplicates("record_id");      attr.append(("One index CRC per patient (earliest ca_seq)", len(d)))
d["age_dx"] = pd.to_numeric(d["age_dx"], errors="coerce")
d = d[d.age_dx >= 20];                                                        attr.append(("Age >= 20", len(d)))
d["time"] = pd.to_numeric(d["tt_os_dx_mos"], errors="coerce")
d["event"] = d["os_dx_status"].map(CODEBOOK["os_dx_status"])
d = d[d.time.notna() & d.event.notna() & (d.time >= 0)];                      attr.append(("Known OS time and status", len(d)))
if HISTOLOGY_MISSING == "exclude":
    d = d[d.naaccr_histology_cd.notna()];                                     attr.append(("Histology recorded (primary analysis)", len(d)))
d["y5"] = np.where(d.time >= HORIZON, 1.0, np.where(d.event == 1, 0.0, np.nan))

# delayed entry: months from diagnosis to the first sequencing report of this cancer
cpt = G["cpt"]; cpt = cpt[cpt.record_id.isin(d.record_id)]
entry = cpt.groupby(["record_id", "ca_seq"])["dx_cpt_rep_mos"].min().rename("entry_mos").reset_index()
d = d.merge(entry, on=["record_id", "ca_seq"], how="left", validate="1:1")
d["entry_mos"] = d["entry_mos"].clip(lower=0)

attrition_g = pd.DataFrame(attr, columns=["Step", "N"]); attrition_g["Removed"] = (-attrition_g.N.diff()).fillna(0).astype(int)
attrition_g.to_csv(OUTG / "genie_attrition.csv", index=False); display(attrition_g)
n_ind = int(d.y5.isna().sum())
print(f"5-year label: survived={int((d.y5 == 1).sum())}  died<5y={int((d.y5 == 0).sum())}  indeterminate (alive, <60 mo)={n_ind} ({100*n_ind/len(d):.1f}%)")
print(f"Delayed entry: median {d.entry_mos.median():.1f} months from diagnosis to sequencing report (IQR {d.entry_mos.quantile(.25):.1f}-{d.entry_mos.quantile(.75):.1f}); "
      f"{int(d.entry_mos.isna().sum())} without a sequencing date | {int((d.entry_mos > d.time).sum())} with entry after end of follow-up (check)")

## Step 6 — Build the SEER-format predictor table from GENIE
Each function converts one GENIE field into the exact SEER column the model expects. Predictors that **do not exist** in GENIE are set to missing
**only for stress tests**, and are listed in the mapping table. CEA follows the SEER definition (*pre-treatment* interpretation): the CEA value closest to
diagnosis inside `CEA_WINDOW_DAYS`, classified as Positive if above the upper normal limit reported with that test. SEER's "borderline" category has no GENIE equivalent.


In [ ]:
def age_band_mid(a):                       # SEER model used the midpoint of 5-year age bands (90+ -> 92)
    a = float(a)
    return 92.0 if a >= 90 else float(int(a // 5) * 5 + 2)

def norm_stage(v):
    s = str(v).strip().upper().replace("STAGE", "").strip()
    if s in ("", "NAN", "99", "88", "UNKNOWN"): return "Unknown"
    if s == "0": return "0"
    if s[:1] in "1234": return {"1": "I", "2": "II", "3": "III", "4": "IV"}[s[0]]
    for r in ("IV", "III", "II", "I"):
        if s.startswith(r): return r
    return "Unknown"

def norm_tnm(v, letter):
    s = str(v).strip().upper()
    if s in ("", "NAN", "88", "NOT APPLICABLE"): return "Unknown"
    s = s.lstrip(letter).lstrip("CPY").lstrip(letter)  # observed forms: Tp3, P3, p3, T3, Np1A, Mc0, pM1b, T (= blank) -> 3, 3, 3, 3, 1A, 0, 1B, ""
    if letter == "T":
        if s.startswith("IS"): return "Tis"
        return {"0": "T0", "1": "T1", "2": "T2", "3": "T3", "4": "T4"}.get(s[:1], "Unknown")
    if letter == "N": return {"0": "N0", "1": "N1", "2": "N2"}.get(s[:1], "Unknown")
    return {"0": "M0", "1": "M1"}.get(s[:1], "Unknown")

def path_else_clin(row, letter):
    p = norm_tnm(row.get(f"naaccr_path_{letter.lower()}_cd"), letter)
    return p if p != "Unknown" else norm_tnm(row.get(f"naaccr_clin_{letter.lower()}_cd"), letter)

def histology_group(code_):
    if pd.isna(code_): return np.nan
    try: h = int(str(int(float(code_)))[:4])           # BPC mixes 4-digit (8140) and 5-digit (81403) codes
    except ValueError: return "Other"
    if 8140 <= h <= 8389: return "adenomas and adenocarcinomas"
    if 8440 <= h <= 8499: return "cystic, mucinous and serous neoplasms"
    if 8010 <= h <= 8049: return "epithelial neoplasms, NOS"
    return "Other"                                   # SEER model grouped histologies < 1% as "Other"; check against meta categories below

HISPANIC = {"Mexican (includes Chicano)", "Spanish surname only", "Spanish NOS or Hispanic NOS or Latino NOS"}     # observed NAACCR ethnicity labels
RACE_MAP = {"White": "NH White", "Black": "NH Black", "Chinese": "NH Asian or Pacific Islander", "Other Asian": "NH Asian or Pacific Islander",
            "Laotian": "NH Asian or Pacific Islander", "Hawaiian": "NH Asian or Pacific Islander",
            "American Indian, Aleutian, or Eskimo": "NH American Indian/Alaska Native", "Unknown": "NH Unknown Race", "Other": "NH Unknown Race"}
def race_seer(row):
    # "Unknown whether Spanish or not" is treated as non-Hispanic (SEER uses an imputation algorithm, NHIA, that GENIE does not have) - limitation
    if str(row.get("naaccr_ethnicity_code")) in HISPANIC: return "Hispanic (All Races)"
    return RACE_MAP.get(str(row.get("naaccr_race_code_primary")), "UNMAPPED")

# CEA (pre-treatment)
tm = G["tm"]; tm = tm[(tm.tm_type == "CEA") & tm.record_id.isin(d.record_id)].copy()
tm = tm[tm.dx_tm_days.between(*CEA_WINDOW_DAYS)].assign(absd=lambda t: t.dx_tm_days.abs()).sort_values("absd")
cea1 = tm.groupby("record_id").first()
ok_ = cea1.tm_num_result.notna() & cea1.tm_normal_range_upper.notna()      # upper normal limit varies by lab (2.5-5 ng/mL)
cea_cat = np.select([ok_ & (cea1.tm_num_result > cea1.tm_normal_range_upper), ok_], ["Positive", "Negative"], "Unknown")
cea_map = pd.Series(cea_cat, index=cea1.index)

X = pd.DataFrame(index=d.index)
X["Age_years"]      = d.age_dx.map(age_band_mid)
X["Sex"]            = d.naaccr_sex_code.map(CODEBOOK["naaccr_sex_code"]).fillna("Unknown")
X["Race"]           = d.apply(race_seer, axis=1)
X["Site"]           = d.ca_d_site.map(SITE_MAP).fillna("Unknown")
X["Histology"]      = d.naaccr_histology_cd.map(histology_group).fillna("adenomas and adenocarcinomas")   # only reached when HISTOLOGY_MISSING == "modal"
X["Grade"]          = d.ca_grade.map(CODEBOOK["ca_grade"]).fillna("Unknown")
X["Stage_Group"]    = d.best_ajcc_stage_cd.map(norm_stage)                                   # best AJCC stage (blank in ~27%)
X.loc[X.Stage_Group.eq("Unknown"), "Stage_Group"] = d.stage_dx.map(CODEBOOK["stage_dx"]).fillna("Unknown")   # fall back to BPC stage at diagnosis
X["T_Stage"]        = d.apply(lambda r: path_else_clin(r, "T"), axis=1)
X["N_Stage"]        = d.apply(lambda r: path_else_clin(r, "N"), axis=1)
X["M_Stage"]        = d.apply(lambda r: path_else_clin(r, "M"), axis=1)
X["Summary_Stage"]  = d.naaccr_seer_sum_stage.map(CODEBOOK["naaccr_seer_sum_stage"]).fillna("Unknown")
X["CEA"]            = d.record_id.map(cea_map).fillna("Unknown")
X["Perineural_Invasion"] = d.ca_crc_peri_inv.map(PNI_MAP).fillna("Unknown")                     # no "Absent" in GENIE -> not usable
td = pd.to_numeric(d.ca_crc_td, errors="coerce")
X["Tumor_Deposits"] = np.select([td.eq(0), td.gt(0) | d.ca_crc_td.astype(str).str.startswith("Tumor Deposits identified")], ["None", "Present"], "Unknown")
NOT_IN_GENIE = {"Marital_Status": "Unknown", "Income_USD": np.nan, "RUCC": np.nan, "Tumor_Size_mm": np.nan,
                "Nodes_Examined": np.nan, "Nodes_Positive": np.nan, "Node_Ratio": np.nan}
for c, v in NOT_IN_GENIE.items(): X[c] = v

## Step 7 — SEER-to-GENIE mapping table and compatibility verdict
Missingness is computed from the GENIE cohort. "Unknown" counts as missing information. Categories that the SEER model never saw are flagged
(the one-hot encoder would silently ignore them, which would change the feature's meaning).


In [ ]:
SOURCE = {"Age_years": ("age_dx", "index", "years -> SEER 5-y band midpoint"), "Sex": ("naaccr_sex_code", "patient", "label"),
          "Race": ("naaccr_race_code_primary + naaccr_ethnicity_code", "patient", "NAACCR -> SEER race/origin recode"),
          "Site": ("ca_d_site", "index", "ICD-O-3 topography -> SEER site recode"), "Histology": ("naaccr_histology_cd", "index", "ICD-O-3 code -> SEER broad group"),
          "Grade": ("ca_grade", "index", "I-IV labels; blank -> Unknown"), "Stage_Group": ("best_ajcc_stage_cd, else stage_dx", "index", "AJCC 6th/7th edition; BPC stage fallback"),
          "T_Stage": ("naaccr_path_t_cd / naaccr_clin_t_cd", "index", "pathologic else clinical"), "N_Stage": ("naaccr_path_n_cd / naaccr_clin_n_cd", "index", "pathologic else clinical"),
          "M_Stage": ("naaccr_path_m_cd / naaccr_clin_m_cd", "index", "pathologic else clinical"), "Summary_Stage": ("naaccr_seer_sum_stage", "index", "NAACCR code -> SEER category"),
          "CEA": ("tm_type == CEA, tm_num_result vs tm_normal_range_upper", "tm", "value -> pre-treatment interpretation; no 'borderline'"),
          "Perineural_Invasion": ("ca_crc_peri_inv", "index", "no 'absent' value in GENIE -> NOT usable"), "Tumor_Deposits": ("ca_crc_td", "index", "count -> None/Present")}
rows = []
for f in sorted(set(FEATS) | set(SOURCE) | set(NOT_IN_GENIE)):
    src = SOURCE.get(f, ("— not available in GENIE BPC CRC public files", "—", "—"))
    s = X[f]; miss = s.isna() | s.astype(str).eq("Unknown")
    unseen = sorted(set(s.dropna().astype(str)) - set(CATEG_SEER.get(f, s.dropna().astype(str)))) if f in CATEG_SEER else []
    usable = (f not in NOT_IN_GENIE) and not unseen and f != "Perineural_Invasion"
    rows.append({"SEER feature": f, "in model": f in FEATS, "GENIE field": src[0], "GENIE file": src[1], "coding / unit difference": src[2],
                 "GENIE % missing/unknown": round(100 * miss.mean(), 1), "unseen categories": ", ".join(unseen), "suitable": "yes" if usable else "NO"})
mapping = pd.DataFrame(rows); mapping.to_csv(OUTG / "seer_genie_feature_mapping.csv", index=False); display(mapping)

missing_required = [f for f in FEATS if f in NOT_IN_GENIE or f == "Perineural_Invasion"]
bad_levels = mapping[(mapping["in model"]) & (mapping["unseen categories"] != "")]["SEER feature"].tolist()
print("Required predictors not available in GENIE:", missing_required or "none")
print("Required predictors with categories unseen in SEER:", bad_levels or "none")
COMPATIBLE = not missing_required and not bad_levels
if not COMPATIBLE and not ALLOW_STRESS_TEST:
    raise SystemExit(f"Model '{MODEL_TO_VALIDATE}' is NOT compatible with GENIE. Use MODEL_TO_VALIDATE='core', fix the codebook, or set ALLOW_STRESS_TEST=True (report as a stress test, not external validation).")
ANALYSIS_LABEL = "external validation" if COMPATIBLE else "STRESS TEST (missing predictors) - not an external validation"
print("Analysis type:", ANALYSIS_LABEL)

## Step 8 — Apply the frozen model (prediction only)
`predict_proba` only — no `fit`. The pre-specified threshold is applied to the uncalibrated probability exactly as in SEER; the isotonic calibrator (fitted on SEER) gives the probabilities used for calibration plots.


In [ ]:
p_raw = model.predict_proba(X[FEATS])[:, 1]
p_cal = calibrator.predict(p_raw) if calibrator is not None else p_raw
d["p_surv5_raw"], d["p_surv5"], d["pred5"] = p_raw, p_cal, (p_raw >= THRESHOLD).astype(int)
print(f"Predicted 5-year survival (calibrated): mean {p_cal.mean():.3f}, median {np.median(p_cal):.3f}")

## Step 9 — Performance on the GENIE cohort
1. **Binary 5-year metrics** on patients with a determinate label: ROC-AUC, PR-AUC, sensitivity, specificity, PPV, NPV, F1, Brier, calibration intercept and slope, all with 95 % bootstrap CIs; repeated in the subset sequenced ≤ 6 months after diagnosis (minimal delayed entry).
2. **Time-to-event metrics** on everyone: Harrell's C-index (risk = 1 − p), and calibration of predicted vs **Kaplan–Meier observed 5-year survival with delayed entry** (entry = first sequencing report) by risk tertile.


In [ ]:
def calib_int_slope(y, p):
    y = np.asarray(y, float); p = np.clip(np.asarray(p, float), 1e-6, 1 - 1e-6); lp = np.log(p / (1 - p)); a = 0.0
    for _ in range(100):
        mu = 1 / (1 + np.exp(-(a + lp))); st = np.sum(y - mu) / max(np.sum(mu * (1 - mu)), 1e-12); a += st
        if abs(st) < 1e-10: break
    X1 = np.column_stack([np.ones_like(lp), lp]); b = np.zeros(2)
    for _ in range(100):
        mu = 1 / (1 + np.exp(-X1 @ b)); W = mu * (1 - mu); st = np.linalg.solve(X1.T @ (X1 * W[:, None]) + 1e-9 * np.eye(2), X1.T @ (y - mu)); b += st
        if np.max(np.abs(st)) < 1e-10: break
    return a, b[1]

def metrics(y, p_raw_, p_cal_):
    pred = (p_raw_ >= THRESHOLD).astype(int); tp = np.sum((pred == 1) & (y == 1)); tn = np.sum((pred == 0) & (y == 0)); fp = np.sum((pred == 1) & (y == 0)); fn = np.sum((pred == 0) & (y == 1))
    a, s = calib_int_slope(y, p_cal_)
    return {"n": len(y), "events (died <5y)": int((y == 0).sum()), "ROC-AUC": roc_auc_score(y, p_raw_), "PR-AUC": average_precision_score(y, p_raw_),
            "Sensitivity": tp / max(tp + fn, 1), "Specificity": tn / max(tn + fp, 1), "PPV": tp / max(tp + fp, 1), "NPV": tn / max(tn + fn, 1),
            "F1-score": 2 * tp / max(2 * tp + fp + fn, 1), "Brier": brier_score_loss(y, p_cal_), "Calib. intercept": a, "Calib. slope": s}

def with_ci(sub, label):
    y = sub.y5.values.astype(int); pr, pc = sub.p_surv5_raw.values, sub.p_surv5.values
    est = metrics(y, pr, pc); rng = np.random.default_rng(SEED); bs = []
    for _ in range(N_BOOT):
        b = rng.integers(0, len(y), len(y))
        if 0 < y[b].sum() < len(y): bs.append(metrics(y[b], pr[b], pc[b]))
    bs = pd.DataFrame(bs)
    return pd.Series({k: (f"{v:.3f} ({bs[k].quantile(.025):.3f}-{bs[k].quantile(.975):.3f})" if k not in ("n", "events (died <5y)") else v) for k, v in est.items()}, name=label)

det = d[d.y5.notna()]
res = [with_ci(det, f"All determinate ({ANALYSIS_LABEL})")]
early = det[det.entry_mos <= 6]
if early.y5.nunique() == 2 and len(early) >= 50: res.append(with_ci(early, "Sequenced <= 6 months after diagnosis"))
res = pd.DataFrame(res); res.to_csv(OUTG / "genie_binary_metrics_95ci.csv"); display(res.T)

# time-to-event: C-index and KM-with-delayed-entry calibration
cidx = concordance_index(d.time, d.p_surv5, d.event)
print(f"Harrell's C-index (all {len(d)} patients, delayed entry not accounted for): {cidx:.3f}")
d["risk_tertile"] = pd.qcut(1 - d.p_surv5, 3, labels=["low risk", "intermediate", "high risk"])
kmf = KaplanMeierFitter(); cal_rows = []; fig, ax = plt.subplots(1, 2, figsize=(13, 4.8))
for g, part in d.groupby("risk_tertile", observed=True):
    ok = part.entry_mos.notna() & (part.entry_mos < part.time)
    kmf.fit(part.time[ok], part.event[ok], entry=part.entry_mos[ok], label=f"{g} (n={int(ok.sum())})")
    kmf.plot_survival_function(ax=ax[0]); obs = float(kmf.predict(HORIZON))
    lo, hi = kmf.confidence_interval_survival_function_.iloc[(np.abs(kmf.confidence_interval_survival_function_.index - HORIZON)).argmin()]
    cal_rows.append((g, int(ok.sum()), part.p_surv5.mean(), obs, lo, hi))
ax[0].axvline(HORIZON, ls=":", c="k"); ax[0].set_title("GENIE: KM by predicted-risk tertile (delayed entry)"); ax[0].set_xlabel("months since diagnosis")
calk = pd.DataFrame(cal_rows, columns=["tertile", "n", "mean predicted S(60)", "KM observed S(60)", "KM 95% low", "KM 95% high"])
ax[1].errorbar(calk["mean predicted S(60)"], calk["KM observed S(60)"], yerr=[calk["KM observed S(60)"] - calk["KM 95% low"], calk["KM 95% high"] - calk["KM observed S(60)"]], fmt="o")
ax[1].plot([0, 1], [0, 1], "k--"); ax[1].set_xlabel("predicted 5-y survival"); ax[1].set_ylabel("observed (KM, delayed entry)"); ax[1].set_title("5-year calibration by tertile")
plt.tight_layout(); savefig("genie_km_calibration"); plt.show()
calk.to_csv(OUTG / "genie_km_calibration.csv", index=False); display(calk.round(3))

if det.y5.nunique() == 2:
    fp_, mp_ = calibration_curve(det.y5, det.p_surv5, n_bins=min(10, max(3, len(det) // 50)), strategy="quantile")
    plt.figure(figsize=(5, 5)); plt.plot(mp_, fp_, "o-"); plt.plot([0, 1], [0, 1], "k--"); plt.xlabel("predicted"); plt.ylabel("observed (determinate patients)")
    plt.title("Calibration (binary 5-year)"); plt.tight_layout(); savefig("genie_calibration_binary"); plt.show()

## Step 10 — Subgroups and the validation report
Performance by stage group and by institution (case-mix differs strongly between BPC centres). Then a short machine-generated report with the counts and numbers to copy into the paper. **No patient-level rows are written.**


In [ ]:
sg = []
for col in ["Stage_Group", "institution"]:
    key = X[col] if col in X else d[col]
    for lvl, idx_ in det.groupby(key.loc[det.index]).groups.items():
        part = det.loc[idx_]
        if len(part) >= 30 and part.y5.nunique() == 2:
            sg.append((col, lvl, len(part), roc_auc_score(part.y5, part.p_surv5_raw), part.y5.mean(), part.p_surv5.mean()))
sg = pd.DataFrame(sg, columns=["subgroup", "level", "n", "ROC-AUC", "observed 5-y survival", "mean predicted"]); sg.to_csv(OUTG / "genie_subgroups.csv", index=False); display(sg.round(3))

report = f"""# GENIE BPC CRC v2.0 - validation report
* Model: {MODEL_TO_VALIDATE} ({model_path.name}, sha256 {sha[:16]}...) | analysis type: **{ANALYSIS_LABEL}**
* Required predictors missing in GENIE: {missing_required or 'none'}
* Cohort: {len(d)} eligible index CRC cancers ({attrition_g.N.iloc[0]} before exclusions); determinate 5-year label: {len(det)}; indeterminate (alive, <60 mo): {n_ind}
* Delayed entry: median {d.entry_mos.median():.1f} months from diagnosis to sequencing report
* Binary 5-year metrics: see genie_binary_metrics_95ci.csv
* Harrell's C-index: {cidx:.3f}
* Limitations: left truncation (entry at sequencing), referral-centre case mix, missing SEER predictors {list(NOT_IN_GENIE)}, coding differences (AJCC edition, CEA definition).
"""
(OUTG / "GENIE_validation_report.md").write_text(report); print(report)